In [1]:
import os
import time

from dotenv import load_dotenv
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from msrest.authentication import CognitiveServicesCredentials
from PIL import Image, ImageDraw
from sklearn.metrics import accuracy_score, precision_score, recall_score

load_dotenv()
ENDPOINT = os.getenv("AZURE_ENDPOINT")
API_KEY = os.getenv("AZURE_API_KEY")

BASE_IMAGE_PATH = r"C:\Laborator AI\Laborator 3\images"
OUTPUT_PATH = os.path.join(BASE_IMAGE_PATH, "output")

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

client = ComputerVisionClient(ENDPOINT, CognitiveServicesCredentials(API_KEY))

manual_annotations = {
    'bike1.jpg': [[5, 30, 403, 376]],
    'bike02.jpg': [[17, 86, 362, 236]],
    'bike03.jpg': [[62, 144, 135, 250], [155, 140, 191, 257]], # 2 biciclete
    'bike04.jpg': [[0, 0, 415, 415]],
    'bike05.jpg': [[68, 39, 289, 306]],
    'bike06.jpg': [[62, 144, 135, 250], [155, 140, 191, 257]], # 2 biciclete
    'bike07.jpg': [[60, 185, 240, 230]],
    'bike08.jpg': [[42, 8, 346, 342]],
    'bike09.jpg': [[5, 7, 375, 373]],
    'bike10.jpg': [[140, 124, 237, 281]]
}

def calculate_iou(boxA, boxB):
    xA_min, yA_min, xA_max, yA_max = boxA[0], boxA[1], boxA[0]+boxA[2], boxA[1]+boxA[3]
    xB_min, yB_min, xB_max, yB_max = boxB[0], boxB[1], boxB[0]+boxB[2], boxB[1]+boxB[3]

    xI_min = max(xA_min, xB_min)
    yI_min = max(yA_min, yB_min)
    xI_max = min(xA_max, xB_max)
    yI_max = min(yA_max, yB_max)

    inter_width = max(0, xI_max - xI_min)
    inter_height = max(0, yI_max - yI_min)
    inter_area = inter_width * inter_height

    boxA_area = boxA[2] * boxA[3]
    boxB_area = boxB[2] * boxB[3]

    union_area = boxA_area + boxB_area - inter_area
    if union_area == 0:
        return 0
    return inter_area / float(union_area)

def evaluate_multiple_detections(pred_boxes, gt_boxes, iou_threshold=0.5):
    tp = 0
    fp = 0

    matched_gt = [False] * len(gt_boxes)
    image_ious = []

    for pred_box in pred_boxes:
        best_iou = 0
        best_gt_idx = -1

        for idx, gt_box in enumerate(gt_boxes):
            iou = calculate_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx

        image_ious.append(best_iou)

        if best_iou >= iou_threshold:
            if not matched_gt[best_gt_idx]:
                tp += 1
                matched_gt[best_gt_idx] = True # Îl marcăm ca găsit
            else:
                fp += 1 # A mai fost găsit o dată de alt chenar AI (duplicat)
        else:
            fp += 1 # AI-ul a detectat ceva greșit

    # False Negatives sunt bicicletele manuale care au rămas nemarcate (false in matched_gt)
    fn = matched_gt.count(False)

    return tp, fp, fn, image_ious

def process_images():
    print("Începem procesarea imaginilor...")

    y_true_classification = []
    y_pred_classification = []

    total_tp = 0
    total_fp = 0
    total_fn = 0
    all_ious = []

    for root, dirs, files in os.walk(BASE_IMAGE_PATH):
        if "output" in root:
            continue

        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                is_bike_folder = "bikes" in root.lower()

                y_true_classification.append(1 if is_bike_folder else 0)

                print(f"\nProcesez imaginea: {file}")

                with open(img_path, "rb") as image_stream:
                    analysis = client.analyze_image_in_stream(
                        image_stream,
                        visual_features=['Tags', 'Objects']
                    )

                time.sleep(3)

                tags = [tag.name.lower() for tag in analysis.tags]
                has_bike_ai = 1 if ('bicycle' in tags or 'bike' in tags) else 0
                y_pred_classification.append(has_bike_ai)

                detected_boxes = []
                img_to_draw = Image.open(img_path)
                draw = ImageDraw.Draw(img_to_draw)

                for obj in analysis.objects:
                    if obj.object_property.lower() in ['bicycle', 'bike']:
                        rect = obj.rectangle
                        box = [rect.x, rect.y, rect.w, rect.h]
                        detected_boxes.append(box)

                        draw.rectangle(
                            [(rect.x, rect.y), (rect.x + rect.w, rect.y + rect.h)],
                            outline="red",
                            width=3
                        )

                output_file = os.path.join(OUTPUT_PATH, f"ai_detected_{file}")
                img_to_draw.save(output_file)
                print(f" -> Imagine salvată cu detectii AI: {output_file}")

                if file in manual_annotations:
                    manual_boxes = manual_annotations[file]

                    tp, fp, fn, img_ious = evaluate_multiple_detections(detected_boxes, manual_boxes)

                    total_tp += tp
                    total_fp += fp
                    total_fn += fn
                    all_ious.extend(img_ious)

                    print(f" -> Detecții pt {file}: {tp} TP, {fp} FP, {fn} FN (IoU mediu: {sum(img_ious)/len(img_ious) if img_ious else 0:.4f})")

    print("REZULTATE")
    print("="*40)

    print("\n1. Poze cu bicicletă vs. fără bicicletă:")
    acc = accuracy_score(y_true_classification, y_pred_classification)
    prec = precision_score(y_true_classification, y_pred_classification, zero_division=0)
    rec = recall_score(y_true_classification, y_pred_classification, zero_division=0)
    print(f" - Acuratețe (Accuracy): {acc:.2f}")
    print(f" - Precizie (Precision): {prec:.2f}")
    print(f" - Recall: {rec:.2f}")

    print("\n2. Performanța Detecției (AI vs Manual):")
    if (total_tp + total_fp + total_fn) > 0:
        detection_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
        detection_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
        mean_iou = sum(all_ious) / len(all_ious) if all_ious else 0

        print(f"Total Adevărat Pozitive (Biciclete găsite corect): {total_tp}")
        print(f"Total Fals Pozitive (Alarme false / chenare greșite): {total_fp}")
        print(f"Total Fals Negative (Biciclete ratate de AI): {total_fn}\n")

        print(f" - Precizia Detecției: {detection_precision:.2f} (din ce a detectat AI-ul, {detection_precision*100:.0f}% a fost corect)")
        print(f" - Recall Detecție:    {detection_recall:.2f} (din totalul de biciclete reale, AI-ul a găsit {detection_recall*100:.0f}%)")
        print(f" - IoU Mediu (mIoU):   {mean_iou:.4f}")
    else:
        print("Nu există date pentru a calcula performanța detecției.")

process_images()

Începem procesarea imaginilor...

Procesez imaginea: animals.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_animals.png

Procesez imaginea: binClass.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_binClass.png

Procesez imaginea: cm.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_cm.png

Procesez imaginea: IoU.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_IoU.png

Procesez imaginea: IoU2.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_IoU2.png

Procesez imaginea: objectDetection.png
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_objectDetection.png

Procesez imaginea: people.jpg
 -> Imagine salvată cu detectii AI: C:\Laborator AI\Laborator 3\images\output\ai_detected_people.jpg

Procesez imaginea: PR.png
 -> Imagin